# Cross-branch DPO-delta transfer — Stage 0 + Stage 1 gate

**Scope of this notebook, deliberately narrow:** Stage 0 (micro-validation, ~8 prompts) and the 8-unit Stage-1 gate (`baseline_target`, `reference_target`, `own_delta_target` × {0.5, 1.0, 2.0}, `own_normmatched_random` × {0.5, 1.0, 2.0}). Runs on quadrant A/D held-out + full B + full C (414 prompts per condition).

**This notebook does NOT run Stage 2** (cross-branch identity transfer, direction arms, dose-matched arms, Procrustes, etc.). Those conditions are declared in the code but refused by the worker without an explicit `--allow-stage2` flag this notebook never passes. Stage 1 is a hard gate — whether Stage 2 is worth running at all is a decision to make *after* reading this notebook's output, not before.

**Estimated runtime:** activation preflight + delta assembly are CPU/seconds. Stage 0 is a handful of generations. The 8-unit gate is comparable to one existing v2 steering pass — well under a single T4 session.

Run cells top to bottom. Stop and read before the final "DO NOT PROCEED" cell.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone and pin the exact commit

Pins the same commit the crossbranch code was developed and tested against. The crossbranch package itself is not yet pushed to GitHub — it's applied as a small patch in the next section, uploaded from your machine, so nothing is committed or pushed on your behalf.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = '2c4f53b1b34c7e1673e8111491114f86d2f1ac40'

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", PINNED_COMMIT], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert commit == PINNED_COMMIT, f"wrong commit: {commit} != {PINNED_COMMIT}"
print("checked out", commit)

## 3. Apply the crossbranch patch

Upload `crossbranch_p0_patch.zip` when prompted (built locally from the audited, 134-test-covered `src/analysis/crossbranch/` + `tests/analysis/crossbranch/` package). This only adds new files under those two directories — it touches nothing that was already in the clone.

In [ ]:
from google.colab import files
import zipfile, io

uploaded = files.upload()  # select crossbranch_p0_patch.zip
assert len(uploaded) == 1, "upload exactly one file: crossbranch_p0_patch.zip"
name, data = next(iter(uploaded.items()))

with zipfile.ZipFile(io.BytesIO(data)) as z:
    names = z.namelist()
    assert all(n.startswith(('src/analysis/crossbranch/', 'tests/analysis/crossbranch/')) for n in names), (
        "patch contains files outside the crossbranch package — refusing to apply"
    )
    z.extractall('.')

print(f"applied {len(names)} files from {name}")
for n in sorted(names):
    print(" ", n)

## 4. Install dependencies, check GPU

In [ ]:
!pip -q install -r requirements.txt
# Colab preinstalls torchao; it breaks transformers on quantized/8-bit loads.
!pip uninstall -y torchao || true
!nvidia-smi

## 5. Get the four bound activation arrays into `results/activations/`

Crossbranch needs `M2`, `M3`, `M2_alt`, `M3_alt` `_final.npy` + `_pooled.npy` + `_metadata.json` + `_metadata_binding.json`, bound to the frozen 654-row benchmark. These already exist in your Google Drive (they're what the `results-...zip` parts you downloaded were exported from) — point `RESULTS_SOURCE_DIR` at that folder so this notebook copies them across the Drive mount, no re-upload or re-extraction needed.

If you're not sure of the path, the next cell searches your Drive for candidates.

In [ ]:
# Helper: list folders under MyDrive that look like a `results` export
# (contain an `activations` subfolder). Read-only, just to help you fill in
# RESULTS_SOURCE_DIR below.
import subprocess
out = subprocess.run(
    ['find', '/content/drive/MyDrive', '-maxdepth', '4', '-type', 'd', '-name', 'activations'],
    capture_output=True, text=True,
)
print(out.stdout or "(no 'activations' folder found under MyDrive at depth <=4 — widen the search or upload manually)")

In [ ]:
# EDIT THIS: the folder that CONTAINS 'activations/' (i.e. the parent of the
# 'results' directory you downloaded as a zip). Example:
#   RESULTS_SOURCE_DIR = '/content/drive/MyDrive/dpo_safety_v2/results'
RESULTS_SOURCE_DIR = '/content/drive/MyDrive/PASTE_PATH_HERE/results'

import shutil
from pathlib import Path

src_act = Path(RESULTS_SOURCE_DIR) / 'activations'
assert src_act.exists(), (
    f"{src_act} does not exist — fix RESULTS_SOURCE_DIR above, or use the manual-upload "
    "fallback cell instead"
)

dst_act = Path('results/activations')
dst_act.mkdir(parents=True, exist_ok=True)

STAGES = ('M2', 'M3', 'M2_alt', 'M3_alt')
SUFFIXES = ('_final.npy', '_pooled.npy', '_metadata.json', '_metadata_binding.json')

copied, missing = [], []
for stage in STAGES:
    for suf in SUFFIXES:
        s = src_act / f'{stage}{suf}'
        if s.exists():
            shutil.copy2(s, dst_act / f'{stage}{suf}')
            copied.append(s.name)
        else:
            missing.append(s.name)

print(f"copied {len(copied)} files")
if missing:
    print(f"MISSING ({len(missing)}), fix RESULTS_SOURCE_DIR or upload these manually:")
    for m in missing:
        print("  ", m)

### 5b. Fallback: manual upload

Only run this if section 5 couldn't find the files on Drive. Upload the two
`results-...zip` parts directly; this extracts just the four needed stages,
same as the local verification already run on this project.

In [ ]:
# Only run if needed.
from google.colab import files
import zipfile

STAGES = ('M2', 'M3', 'M2_alt', 'M3_alt')
SUFFIXES = ('_final.npy', '_pooled.npy', '_metadata.json', '_metadata_binding.json')
wanted = {f'results/activations/{s}{suf}' for s in STAGES for suf in SUFFIXES}

uploaded = files.upload()  # select both results-...-001.zip and -002.zip
for name in uploaded:
    with zipfile.ZipFile(name) as z:
        present = [n for n in z.namelist() if n in wanted]
        z.extractall('.', members=present)
        print(f"{name}: extracted {len(present)} needed files")

## 6. Activation preflight

Same check the `runner --dry-run` performs: shape, row order against the frozen benchmark's identity snapshot, and both SHAs in the binding sidecar. Must report **PASS** for all four stages before continuing — if it doesn't, fix section 5 rather than proceeding.

In [ ]:
import json
from pathlib import Path
import numpy as np
from src.v2_io import load_run_inputs, identity_snapshot, load_json

bp, bsha, sp, ssha = load_run_inputs(None, None, 'logs/direction_split_manifest.json')
rows = [json.loads(l) for l in Path(bp).read_text(encoding='utf-8').splitlines() if l.strip()]
snap = identity_snapshot(rows)
print(f"benchmark sha: {bsha}\nsplit sha:     {ssha}\nrows:          {len(rows)}\n")

act = Path('results/activations')
all_pass = True
for stage in ('M2', 'M3', 'M2_alt', 'M3_alt'):
    arr = np.load(act / f'{stage}_final.npy', mmap_mode='r')
    meta = load_json(act / f'{stage}_metadata.json')
    bind = load_json(act / f'{stage}_metadata_binding.json')
    order_ok = meta == snap
    bind_ok = bind.get('benchmark_sha256') == bsha and bind.get('split_manifest_sha256') == ssha
    ok = arr.shape[0] == len(rows) and order_ok and bind_ok
    all_pass &= ok
    print(f"{stage:8s} shape={str(arr.shape):18s} order={order_ok!s:5s} binding={bind_ok!s:5s}  {'PASS' if ok else 'FAIL'}")

assert all_pass, "activation preflight FAILED — do not proceed"
print("\nAll four stages PASS.")

## 7. Focused test gate

Runs the crossbranch suite fresh in this environment before spending any GPU time.

In [ ]:
!python -m pytest tests/analysis/crossbranch -q

## 8. Assemble deltas (CPU, seconds)

Builds `Δ_A`, `Δ_B`, the within-quadrant-shuffled `Δ_A`, and the `Δ_B`-norm-matched random — the only artifacts Stage 1 consumes. Refuses if the preflight above didn't actually pass.

In [ ]:
!python -m src.analysis.crossbranch.delta

## 9. Dry-run — confirm all 8 Stage-1 units are runnable

In [ ]:
!python -m src.analysis.crossbranch.runner --dry-run

## 10. Stage 0 — real micro-validation

A small real run (8 quadrant-A prompts, `own_delta_target`, coef 1.0, `last_prompt_only`) to verify the protocol on this checkpoint before spending real GPU time on the full gate: tokenization/template match, left padding, the hook site, injection only at the final prompt position during prefill, decode as a strict no-op, and clean row/record_id output.

In [ ]:
!python -m src.analysis.crossbranch.worker \
    --condition own_delta_target --coef 1.0 \
    --quadrants A --limit 8 --force

In [ ]:
# Manual inspection: read the 8 rows and eyeball them before trusting anything downstream.
import json
from pathlib import Path

candidates = sorted(Path('results/crossbranch/raw').glob('crossbranch_AtoB_own_delta_target_coef1*.json'))
assert candidates, "Stage-0 output not found — check the previous cell's output for errors"
rows = json.loads(candidates[-1].read_text(encoding='utf-8'))
print(f"{len(rows)} rows in {candidates[-1].name}\n")
for r in rows:
    print(f"[{r['record_id']}] quadrant={r['quadrant']} coef={r['coef']} inject_mode={r['inject_mode']}")
    print(f"  prompt:   {r['prompt'][:100]}")
    print(f"  response: {r['response'][:200]!r}\n")

**Stop and read the 8 rows above.** Confirm the responses look like real, on-topic completions (not garbage/empty/repeated tokens), the `record_id`s match the quadrant-A prompts you'd expect, and `inject_mode` reads `last_prompt_only`. Only continue once this looks right.

## 11. Stage 1 — the 8-unit gate

Runs `baseline_target`, `reference_target`, `own_delta_target` × {0.5, 1.0, 2.0}, `own_normmatched_random` × {0.5, 1.0, 2.0} on all 414 held-out/full-B/full-C prompts. This is the real GPU time this notebook spends.

In [ ]:
!python -m src.analysis.crossbranch.runner --force

## 12. Gate analysis

In [ ]:
!python -m src.analysis.crossbranch.analyze

In [ ]:
import json
from pathlib import Path

path = Path('results/crossbranch/analysis/crossbranch_AtoB_analysis.json')
result = json.loads(path.read_text(encoding='utf-8'))
g = result['gate']

print(f"gate quadrant (q*)          : {g['gate_quadrant']}")
print(f"max TV(baseline, reference) : {g['max_tv_baseline_to_reference']:.4f}")
print(f"no_target_shift             : {g['no_target_shift']}")
print(f"below_one_row_resolution    : {g['below_one_row_resolution']}")
print(f"tied_quadrants              : {g['tied_quadrants']}")
print()
print(f"mechanical_gate_passed      : {g['mechanical_gate_passed']}")
print(f"passing_coefficients        : {g['passing_coefficients']}")
print(f"inconclusive_by_collapse    : {g['inconclusive_by_collapse']}")
print(f"target_degeneracy_warning   : {g['target_degeneracy_warning']}")
print(f"null_is_not_mechanistic     : {g['null_is_not_mechanistic']}")
print(f"behaviorally_interpretable  : {g['behaviorally_interpretable']}")
print()
print(g['_note'])

## 13. Package results to bring back

Zips everything under `results/crossbranch/` (raw generations, deltas, analysis, manifests — not the large `.npz` deltas' source activations) for download.

In [ ]:
import shutil
shutil.make_archive('/content/crossbranch_stage1_results', 'zip', 'results/crossbranch')

from google.colab import files
files.download('/content/crossbranch_stage1_results.zip')

---
## STOP — DO NOT PROCEED TO STAGE 2

This notebook's scope ends here. Bring `crossbranch_stage1_results.zip` back for review.

- If `mechanical_gate_passed` is `True`: the within-branch delta moved behaviour toward B3 more than a norm-matched random perturbation, at ≥ 1 coefficient, in the predeclared gate quadrant. This is what makes Stage 2 (cross-branch transfer) worth running — it is still a separate decision, not automatic.
- If `False` and `no_target_shift` is `True`: there was no post-DPO behavioural shift in `q*` to reproduce — report this plainly, it is not evidence the delta failed.
- If `False` and `inconclusive_by_collapse` is `True`: every coefficient collapsed into degenerate generation — not a mechanistic null either.
- If `False` otherwise: write the fallback result — *"Under the pre-specified L24 / final-prompt-position additive intervention protocol, the measured DPO-induced delta was not sufficient to reproduce the observed post-DPO behavioral shift."* Do not escalate to Stage 2, Procrustes, or an operator.